In [ ]:
from fastcore.all import *
from iomeval.readers import find_eval, load_evals, eval_url, get_report_urls
from mistocr.core import read_pgs
from pathlib import Path
import json

import pandas as pd
from iomeval.pipeline import run_pipeline, batch_run

pd.set_option('display.max_colwidth', None)

In [ ]:
DATA_PATH = Path().home() / 'iomeval/data'

In [ ]:
EVALS_PATH = Path().home() / 'iomeval/nbs/files/test/evaluations.json'

In [ ]:
evals = load_evals(EVALS_PATH)

In [ ]:
len((DATA_PATH / 'results').ls())

570

In [ ]:
rpt = (DATA_PATH / 'results').ls()[1]
rpt

Path('/app/data/iomeval/data/results/4e876fcb0154da174141c7fc7440cfee.json')

In [ ]:
def is_mapped(o): return bool(o.read_json().get('mappings'))
def is_ready(o): return o.read_json().get('curation_status') == 'sections_selected'
def get_rpts(rpts_path, filt=None):
    rpts = rpts_path.ls()
    if filt: rpts = L(o for o in rpts if filt(o))
    return rpts.sorted(key=lambda x: x.read_json()['meta'].get('Date of Publication', ''), reverse=True)

In [ ]:
path = DATA_PATH / 'results'

In [ ]:
# reports already tagged
len(get_rpts(path, is_mapped))

526

In [ ]:
# reports not yet tagged
len(get_rpts(path, lambda o: not is_mapped(o)))

44

In [ ]:
# ready to be tagged
len(get_rpts(path, lambda o: not is_mapped(o) and is_ready(o)))

4

In [ ]:
# tagged but also ready (for re-tagging?)
len(get_rpts(path, lambda o: is_mapped(o) and is_ready(o)))

526

In [ ]:
def load_result(eval_id, base_path):
    "Load a result JSON by eval_id from base_path"
    return (Path(base_path)/f'{eval_id}.json').read_json()


In [ ]:
rpt.read_json()

{'id': '4e876fcb0154da174141c7fc7440cfee',
 'report_url': 'https://evaluation.iom.int/sites/g/files/tmzbdl151/files/docs/resources/25_%2010_2020_SD10%20TC.0970_Final%20Evaluation%20Report_0.pdf',
 'meta': {'Title': 'EX– POST EVALUATION OF THE PROJECT “PROVIDING TECHNICAL SUPPORT TO IMPROVE BORDER AND MIGRATION MANAGEMENT IN SUDAN”',
  'Year': 2020,
  'Author': 'Theogene Nshimiyimana',
  'Best Practicesor Lessons Learnt': 'Yes',
  'Date of Publication': '2020-04-07',
  'Donor': 'IOM Development Fund',
  'Evaluation Brief': 'Yes',
  'Evaluation Commissioner': 'IOM',
  'Evaluation Coverage': 'Country',
  'Evaluation Period From Date': '2016-12-01',
  'Evaluation Period To Date': '2018-11-30',
  'Executive Summary': 'Yes',
  'External Version of the Report': 'No',
  'Languages': 'English',
  'Migration Thematic Areas': 'Border and identity solutions (border management, security, border assessment etc)',
  'Name of Project(s) Being Evaluated': nan,
  'Number of Pages Excluding annexes': 23.

I'd like to fix all .json files in `(DATA_PATH / 'results')` in the following ways:

- 'Regions Covered': 'RO for East Horn and Southern Africa' is now a string possibly with comma-separated values
- I'd like to modify them in-place to an array [region1, region2], as for 'Countries Covered'

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Let's first check the range of values for `Regions Covered` across all files to make sure we understand what we're dealing with:

```python
from collections import Counter
rpts = (DATA_PATH / 'results').ls()
Counter(r.read_json()['meta'].get('Regions Covered') for r in rpts).most_common(20)
```

<details class='token-usage-details'><summary>$0.1118</summary>

`total=21,599 | in=21,482 | out=117 | cached=0.0% | cache_new=11,142 | searches=0 | $0.1118`

</details>


In [ ]:
from collections import Counter
rpts = (DATA_PATH / 'results').ls()
Counter(r.read_json()['meta'].get('Regions Covered') for r in rpts).most_common(20)

[('RO for East Horn and Southern Africa', 105),
 ('RO for Latin America and the Caribbean', 101),
 ('RO for Europe and Central Asia', 82),
 ('RO for Asia and the Pacific', 72),
 ('RO for West and Central Africa', 66),
 ('Global', 65),
 ('RO for Middle East and North Africa', 43),
 ('', 5),
 ('RO for Middle East and North Africa, RO for West and Central Africa', 4),
 ('RO for Middle East and North Africa, RO for East Horn and Southern Africa',
  4),
 ('RO for Middle East and North Africa, RO for West and Central Africa, RO for East Horn and Southern Africa',
  2),
 ('RO for West and Central Africa, RO for Europe and Central Asia', 2),
 ('RO for East Horn and Southern Africa, RO for Asia and the Pacific', 2),
 ('RO for East Horn and Southern Africa, RO for Latin America and the Caribbean',
  2),
 ('RO for Middle East and North Africa, RO for Europe and Central Asia', 2),
 ('RO for East Horn and Southern Africa, RO for Middle East and North Africa',
  2),
 ('RO for Asia and the Pacific, R

Maybe we can check how many regions we do have (once the string splitted by ',')?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

```python
Counter(r.strip() for rpt in rpts for r in rpt.read_json()['meta'].get('Regions Covered', '').split(',') if r.strip())
```

<details class='token-usage-details'><summary>$0.0676</summary>

`total=22,185 | in=22,112 | out=73 | cached=50.4% | cache_new=10,298 | searches=0 | $0.0676`

</details>


In [ ]:
Counter(r.strip() for rpt in rpts for r in rpt.read_json()['meta'].get('Regions Covered', '').split(',') if r.strip())

Counter({'RO for East Horn and Southern Africa': 122,
         'RO for Latin America and the Caribbean': 108,
         'RO for Europe and Central Asia': 91,
         'RO for Asia and the Pacific': 80,
         'RO for West and Central Africa': 79,
         'Global': 66,
         'RO for Middle East and North Africa': 62,
         'HQ Geneva': 1})

In [ ]:
def fix_regions(path):
    d = path.read_json()
    regions = d['meta'].get('Regions Covered', '')
    if isinstance(regions, list): return
    d['meta']['Regions Covered'] = [r.strip() for r in regions.split(',') if r.strip()]
    path.write_text(json.dumps(d, indent=2, ensure_ascii=False))

In [ ]:
import json
fix_regions(rpt)
print(rpt.read_text()[:300])

{
  "id": "4e876fcb0154da174141c7fc7440cfee",
  "report_url": "https://evaluation.iom.int/sites/g/files/tmzbdl151/files/docs/resources/25_%2010_2020_SD10%20TC.0970_Final%20Evaluation%20Report_0.pdf",
  "meta": {
    "Title": "EX– POST EVALUATION OF THE PROJECT “PROVIDING TECHNICAL SUPPORT TO IMPROVE


In [ ]:
def fix_folder(path):
    for p in path.ls(file_exts=['.json']): fix_regions(p)

In [ ]:
fix_folder(DATA_PATH / 'results')
Counter(type(r.read_json()['meta'].get('Regions Covered')).__name__ for r in rpts)

Counter({'list': 570})

In [ ]:
fix_folder(Path('../../../tagapp/tagapp/data'))

In [ ]:
# To fix indentation
#import json
#rpt.write_text(json.dumps(rpt.read_json(), indent=2, ensure_ascii=False))
#print(rpt.read_text()[:300])

{
  "id": "4e876fcb0154da174141c7fc7440cfee",
  "report_url": "https://evaluation.iom.int/sites/g/files/tmzbdl151/files/docs/resources/25_%2010_2020_SD10%20TC.0970_Final%20Evaluation%20Report_0.pdf",
  "meta": {
    "Title": "EX– POST EVALUATION OF THE PROJECT “PROVIDING TECHNICAL SUPPORT TO IMPROVE
